# Analysing Financial Data

In [ ]:
import matplotlib.pyplot as plt
# Import libraries
import pandas as pd

In [ ]:
df = pd.read_excel('operations.xls')
# !pip install xlrd

In [ ]:
df['is_expense'] = df['Сумма операции'] < 0

In [ ]:
expenses = df[df['is_expense']]

In [ ]:
df.rename(columns={'Дата операции': 'date', 'Сумма операции': 'amount', 'описание': 'description', 'Дата платежа' : 'payment_date', 'Статус' : 'status', 'Валюта операции' : 'currency', 'Сумма платежа' : 'sum', 'Категория' : 'category', 'Описание' : 'description'}, inplace=True)

In [ ]:
df.drop('payment_date', axis=1, inplace=True)
df.drop('Валюта платежа', axis=1, inplace=True)
df.drop('currency', axis=1, inplace=True)
df.drop('Номер карты', axis=1, inplace=True)
df.drop('Кэшбэк', axis=1, inplace=True)
df.drop('Округление на инвесткопилку', axis=1, inplace=True)
df.drop('Бонусы (включая кэшбэк)', axis=1, inplace=True)
df.drop('Сумма операции с округлением', axis=1, inplace=True)
df.drop('MCC', axis=1, inplace=True)
# df.drop('Date', axis=1, inplace=True)
df.drop('sum', axis=1, inplace=True)

In [ ]:
df=df[df['status'] == 'OK']
df.drop('status', axis=1, inplace=True)

In [ ]:
# Convert 'date' column to datetime if it's not already
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
# Set 'date' as the index
# df.set_index('date', inplace=True)

In [ ]:
unique_categories = df['category'].unique()

In [ ]:
#!pip install deep_translator

In [ ]:
# df['category'] = df['category'].map(lambda x: gt.translate(x))

In [ ]:
from deep_translator import GoogleTranslator
gt = GoogleTranslator(source='ru', target='en')
for category in unique_categories:
    print(gt.translate(category))

In [ ]:
#df['category'] = df['category'].map(gt.translate)

In [ ]:
#!pip install seaborn

In [ ]:
# !pip install --upgrade matplotlib
# !pip install --upgrade seaborn

In [ ]:
from deep_translator import GoogleTranslator
def translate_categories(df, category_column='category'):
    # Create translator instance
    translator = GoogleTranslator(source='ru', target='en')

    # Create a dictionary of unique translations to minimize API calls
    unique_categories = df[category_column].unique()
    translation_dict = {
        category: translator.translate(category)
        for category in unique_categories if isinstance(category, str)
    }

    # Replace categories with translations
    df[category_column] = df[category_column].map(translation_dict).fillna(df[category_column])

    return df

In [ ]:
def analyze_spending_patterns(df):
    # Monthly spending by category
    monthly_cat = df.groupby([df['date'].dt.strftime('%Y-%m'), 'category'])['amount'].sum().unstack()

    # Identify irregular large expenses
    def find_outliers(group):
        mean = group['amount'].mean()
        std = group['amount'].std()
        return group[abs(group['amount'] - mean) > 2*std]

    outliers = df.groupby('category').apply(find_outliers)

    # Late night spending (potential impulse purchases)
    if 'time' in df.columns:
        late_night = df[df['time'].dt.hour >= 23]

    # Day of week analysis
    weekday_spending = df.groupby(df['date'].dt.dayofweek)['amount'].mean()

    return monthly_cat, outliers, weekday_spending

In [ ]:
def generate_insights(df):
    # Calculate recurring vs one-time expenses
    recurring = df.groupby(['category', df['date'].dt.strftime('%Y-%m')])['amount'].count()
    recurring = recurring.groupby(level=0).mean()

    # Spending volatility by category
    volatility = df.groupby('category')['amount'].std()

    return recurring, volatility

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

class FinancialAdvisor:
    def __init__(self, df):
        self.df = df
        self.df['date'] = pd.to_datetime(self.df['date'])

    def analyze_spending_patterns(self):
        patterns = {
            'monthly_by_category': self._get_monthly_category_spending(),
            'unusual_expenses': self._find_unusual_expenses(),
            'spending_trends': self._analyze_spending_trends(),
            'recurring_expenses': self._identify_recurring_expenses(),
            'weekend_spending': self._analyze_weekend_spending()
        }
        return patterns

    def _get_monthly_category_spending(self):
        return self.df.groupby([pd.Grouper(key='date', freq='ME'), 'category'])['amount'].sum()

    def _find_unusual_expenses(self):
        def get_outliers(group):
            mean = group['amount'].mean()
            std = group['amount'].std()
            return group[abs(group['amount'] - mean) > 2 * std]

        return self.df.groupby('category').apply(get_outliers)

    def _analyze_spending_trends(self):
        daily_spending = self.df.groupby('date')['amount'].sum()
        return daily_spending.rolling(window=7).mean()

    def _identify_recurring_expenses(self):
        monthly_transactions = self.df.groupby(['category', pd.Grouper(key='date', freq='ME')])['amount'].count()
        return monthly_transactions[monthly_transactions >= 2]

    def _analyze_weekend_spending(self):
        self.df['is_weekend'] = self.df['date'].dt.dayofweek >= 5
        return self.df.groupby('is_weekend')['amount'].agg(['mean', 'sum'])

    def visualize_spending(self):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        # Monthly spending trend
        monthly_total = self.df.groupby(pd.Grouper(key='date', freq='ME'))['amount'].sum()
        axes[0, 0].plot(monthly_total.index, monthly_total.values)
        axes[0, 0].set_title('Monthly Spending Trend')
        axes[0, 0].tick_params(axis='x', rotation=45)

        # Category distribution
        category_totals = self.df.groupby('category')['amount'].sum()
        axes[0, 1].pie(category_totals.values, labels=category_totals.index, autopct='%1.1f%%')
        axes[0, 1].set_title('Spending by Category')

        # Daily spending pattern
        daily_spending = self.df.groupby('date')['amount'].sum()
        axes[1, 0].plot(daily_spending.index, daily_spending.values)
        axes[1, 0].set_title('Daily Spending Pattern')
        axes[1, 0].tick_params(axis='x', rotation=45)

        # Category-wise bar plot
        category_avg = self.df.groupby('category')['amount'].mean()
        axes[1, 1].bar(category_avg.index, category_avg.values)
        axes[1, 1].set_title('Average Spending by Category')
        axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        return fig

    def generate_advice(self):
        advice = []
        patterns = self.analyze_spending_patterns()

        # Analyze monthly trends
        monthly_spending = patterns['monthly_by_category']
        recent_months = monthly_spending.last('3M')

        # Check for increasing trends
        for category in recent_months.index.get_level_values(1).unique():
            category_data = recent_months.xs(category, level=1)
            if len(category_data) >= 3 and category_data.is_monotonic_increasing:
                advice.append(f"WARNING: Your spending in {category} has been steadily increasing. "
                              f"Consider setting a budget for this category.")

        # Analyze unusual expenses
        unusual = patterns['unusual_expenses']
        if not unusual.empty:
            recent_unusual = unusual[unusual.date >= (datetime.now() - timedelta(days=30))]
            if not recent_unusual.empty:
                advice.append("Recent unusual expenses detected in categories: " +
                              ", ".join(recent_unusual['category'].unique()))

        # Weekend spending analysis
        weekend_stats = patterns['weekend_spending']
        if weekend_stats.loc[True, 'mean'] > weekend_stats.loc[False, 'mean'] * 1.5:
            advice.append("Your weekend spending is significantly higher than weekday spending. "
                          "Consider planning weekend activities in advance to control costs.")

        return advice

# Usage example:
# """
# # Initialize advisor
# advisor = FinancialAdvisor(df)
#
# # Get financial advice
# advice = advisor.generate_advice()
# for tip in advice:
#     print(tip)
#
# # Visualize spending
# advisor.visualize_spending()
# plt.show()
# """

In [ ]:
# Initialize advisor
advisor = FinancialAdvisor(df)

In [ ]:
# Get financial advice
advice = advisor.generate_advice()
for tip in advice:
    print(tip)

In [ ]:
def get_monthly_category_spending(df):
    # Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Method 1: Pivot table format (categories as columns)
    monthly_pivot = df.pivot_table(
        index=pd.Grouper(key='date', freq='ME'),
        columns='category',
        values='amount',
        aggfunc='sum'
    ).round(2)

    # Method 2: Stacked format (long format)
    monthly_stacked = df.groupby([
        pd.Grouper(key='date', freq='ME'),
        'category'
    ])['amount'].sum().round(2)

    # Method 3: Unstacked format (similar to pivot)
    monthly_unstacked = monthly_stacked.unstack(fill_value=0).round(2)

    # Method 4: As DataFrame with month and category as columns
    monthly_df = df.groupby([
        pd.Grouper(key='date', freq='ME'),
        'category'
    ])['amount'].sum().reset_index()

    return {
        'pivot': monthly_pivot,
        'stacked': monthly_stacked,
        'unstacked': monthly_unstacked,
        'dataframe': monthly_df
    }

In [ ]:
results = get_monthly_category_spending(df)

In [ ]:
results['pivot']

In [ ]:
results['stacked']

In [ ]:
results['unstacked']

In [ ]:
results['dataframe']

In [ ]:
def format_monthly_report(df):
    # Get monthly spending
    monthly = get_monthly_category_spending(df)['unstacked']

    # Add total row and column
    monthly['Total'] = monthly.sum(axis=1)
    monthly.loc['Average'] = monthly.mean()

    # Format numbers to currency
    formatted = monthly.applymap(lambda x: f"${x:,.2f}")

    return formatted

In [ ]:
def analyze_weekday_spending(df):
    # Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Get day of week
    df['weekday'] = df['date'].dt.day_name()

    # Average spending by day of week
    weekday_avg = df.groupby('weekday')['amount'].mean()

    # Median spending by day of week
    weekday_median = df.groupby('weekday')['amount'].median()

    return weekday_avg, weekday_median

In [ ]:
analyze_weekday_spending(df)

In [ ]:
def analyze_spending_trends(df):
    # Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Daily spending
    daily_spending = df.groupby('date')['amount'].sum()

    # 7-day rolling average
    rolling_avg = daily_spending.rolling(window=7).mean()

    return daily_spending, rolling_avg

In [ ]:
analyze_spending_trends(df)

In [ ]:
def identify_recurring_expenses(df):
    # Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Monthly transactions by category
    monthly_transactions = df.groupby([
        'category',
        pd.Grouper(key='date', freq='M')
    ]).size()

    # Identify recurring expenses
    recurring = monthly_transactions[monthly_transactions >= 2]

    return recurring

In [ ]:
identify_recurring_expenses(df)

In [ ]:
def find_unusual_expenses(df):
    # Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Define a function to find outliers
    def find_outliers(group):
        mean = group['amount'].mean()
        std = group['amount'].std()
        return group[(group['amount'] - mean).abs() > 2 * std]

    # Apply the function to each category
    unusual = df.groupby('category').apply(find_outliers)

    return unusual

In [ ]:
find_unusual_expenses(df)